# 如何将 GPT-4o mini 与 RAG 结合——创建服装搭配应用

欢迎来到服装搭配应用（Clothing Matchmaker App）Jupyter Notebook！本项目展示了 GPT-4o mini 模型在分析服装图片并提取颜色、风格和类型等关键特征方面的强大能力。我们应用的核心依赖于 OpenAI 开发的这一先进图像分析模型，它使我们能够准确识别输入服装的特征。

GPT-4o mini 是一个小型模型，它将自然语言处理与图像识别相结合，能够以低延迟理解文本和视觉输入，并基于二者生成回复。

在 GPT-4o mini 模型能力的基础上，我们采用自定义匹配算法和 RAG 技术，在知识库中搜索与所识别特征相搭配的单品。该算法会考虑颜色兼容性和风格一致性等因素，为用户提供合适的推荐。通过本 notebook，我们旨在展示这些技术在构建服装推荐系统中的实际应用。

将 GPT-4o mini 与 RAG（检索增强生成，Retrieval-Augmented Generation）结合使用有以下几个优势：

1. **上下文理解**：GPT-4o mini 可以分析输入图像并理解其上下文，例如图中描绘的物体、场景和活动。这使得它能够在室内设计、烹饪或教育等各个领域提供更准确、更相关的建议或信息。
2. **丰富的知识库**：RAG 将 GPT-4 的生成能力与一个检索组件相结合，该组件可以访问跨越不同领域的大型信息语料库。这意味着系统可以基于从历史事实到科学概念的广泛知识提供建议或见解。
3. **可定制性**：这种方法便于定制，以满足各种应用中特定的用户需求或偏好。无论是根据用户的艺术品味定制建议，还是根据学生的学习水平提供教育内容，系统都可以进行调整，以提供个性化的体验。

总体而言，GPT-4o mini + RAG 方法结合了生成式和基于检索的 AI 技术的优势，为各种时尚相关应用提供了快速、强大且灵活的解决方案。


### 环境设置

首先，我们将安装必要的依赖，然后导入库并编写一些稍后会用到的工具函数。

In [ ]:
%pip install openai --quiet
%pip install tenacity --quiet
%pip install tqdm --quiet
%pip install numpy --quiet
%pip install typing --quiet
%pip install tiktoken --quiet
%pip install concurrent --quiet

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
import tiktoken
import concurrent
from openai import OpenAI
from tqdm import tqdm
from tenacity import retry, wait_random_exponential, stop_after_attempt
from IPython.display import Image, display, HTML
from typing import List

client = OpenAI()

GPT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-large"
EMBEDDING_COST_PER_1K_TOKENS = 0.00013

### 创建嵌入
现在，我们将通过选择一个数据库并为其生成嵌入来建立知识库。这里我使用 data 文件夹中的 `sample_styles.csv` 文件。它是一个包含 `~44K` 个单品的更大数据集的样本。这一步也可以用开箱即用的向量数据库来替代。例如，你可以按照[这些 cookbook](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases) 中的任意一个来设置你的向量数据库。

In [ ]:
styles_filepath = "data/sample_clothes/sample_styles.csv"
styles_df = pd.read_csv(styles_filepath, on_bad_lines='skip')
print(styles_df.head())
print("Opened dataset successfully. Dataset has {} items of clothing.".format(len(styles_df)))

现在我们将为整个数据集生成嵌入。我们可以并行执行这些嵌入的生成，以确保脚本能够扩展到更大的数据集。借助这一逻辑，为完整的 `44K` 条目数据集创建嵌入的时间从约 4 小时缩短到约 2-3 分钟。

In [ ]:
## Batch Embedding Logic

# Simple function to take in a list of text objects and return them as a list of embeddings
@retry(wait=wait_random_exponential(min=1, max=40), stop=stop_after_attempt(10))
def get_embeddings(input: List):
    response = client.embeddings.create(
        input=input,
        model=EMBEDDING_MODEL
    ).data
    return [data.embedding for data in response]


# Splits an iterable into batches of size n.
def batchify(iterable, n=1):
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx : min(ndx + n, l)]
     

# Function for batching and parallel processing the embeddings
def embed_corpus(
    corpus: List[str],
    batch_size=64,
    num_workers=8,
    max_context_len=8191,
):
    # Encode the corpus, truncating to max_context_len
    encoding = tiktoken.get_encoding("cl100k_base")
    encoded_corpus = [
        encoded_article[:max_context_len] for encoded_article in encoding.encode_batch(corpus)
    ]

    # Calculate corpus statistics: the number of inputs, the total number of tokens, and the estimated cost to embed
    num_tokens = sum(len(article) for article in encoded_corpus)
    cost_to_embed_tokens = num_tokens / 1000 * EMBEDDING_COST_PER_1K_TOKENS
    print(
        f"num_articles={len(encoded_corpus)}, num_tokens={num_tokens}, est_embedding_cost={cost_to_embed_tokens:.2f} USD"
    )

    # Embed the corpus
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        
        futures = [
            executor.submit(get_embeddings, text_batch)
            for text_batch in batchify(encoded_corpus, batch_size)
        ]

        with tqdm(total=len(encoded_corpus)) as pbar:
            for _ in concurrent.futures.as_completed(futures):
                pbar.update(batch_size)

        embeddings = []
        for future in futures:
            data = future.result()
            embeddings.extend(data)

        return embeddings
    

# Function to generate embeddings for a given column in a DataFrame
def generate_embeddings(df, column_name):
    # Initialize an empty list to store embeddings
    descriptions = df[column_name].astype(str).tolist()
    embeddings = embed_corpus(descriptions)

    # Add the embeddings as a new column to the DataFrame
    df['embeddings'] = embeddings
    print("Embeddings created successfully.")

#### 创建嵌入的两种方式：
下一行代码将为示例服装数据集**创建嵌入**。处理大约需要 0.02 秒，另外还需要约 30 秒将结果写入本地 .csv 文件。该过程使用我们的 `text_embedding_3_large` 模型，价格为 `$0.00013/1K` tokens。鉴于该数据集约有 `1K` 个条目，下面的操作大约花费 `$0.001`。如果你决定使用包含 `44K` 个条目的完整数据集，此操作的处理时间为 2-3 分钟，大约花费 `$0.07`。

**如果你不想自己创建嵌入**，我们将使用一个预先计算好嵌入的数据集。你可以跳过此单元格，并取消注释下一个单元格中的代码，以加载预先计算好的向量。此操作大约需要 1 分钟将所有数据加载到内存中。

In [ ]:
generate_embeddings(styles_df, 'productDisplayName')
print("Writing embeddings to file ...")
styles_df.to_csv('data/sample_clothes/sample_styles_with_embeddings.csv', index=False)
print("Embeddings successfully stored in sample_styles_with_embeddings.csv")

In [ ]:
# styles_df = pd.read_csv('data/sample_clothes/sample_styles_with_embeddings.csv', on_bad_lines='skip')

# # Convert the 'embeddings' column from string representations of lists to actual lists of floats
# styles_df['embeddings'] = styles_df['embeddings'].apply(lambda x: ast.literal_eval(x))

print(styles_df.head())
print("Opened dataset successfully. Dataset has {} items of clothing along with their embeddings.".format(len(styles_df)))

### 构建匹配算法

在本节中，我们将开发一个基于余弦相似度的检索算法，用于在 dataframe 中查找相似的单品。为此我们将使用自定义的余弦相似度函数。虽然 `sklearn` 库提供了内置的余弦相似度函数，但其 SDK 最近的更新导致了兼容性问题，因此我们自己实现了标准的余弦相似度计算。

如果你已经设置好了向量数据库，可以跳过这一步。大多数标准数据库都自带搜索功能，可以简化本指南中后续的步骤。不过，我们希望展示匹配算法可以根据特定需求进行定制，例如特定的阈值或指定返回的匹配数量。

`find_similar_items` 函数接受四个参数：
- `embedding`：我们想要为其查找匹配项的嵌入。
- `embeddings`：用于搜索最佳匹配项的嵌入列表。
- `threshold`（可选）：此参数指定匹配被视为有效的最低相似度分数。阈值越高，匹配越接近（越好）；阈值越低，返回的单品越多，但它们与初始 `embedding` 的匹配程度可能没那么高。
- `top_k`（可选）：此参数决定返回多少个超过给定阈值的单品。它们将是与所提供 `embedding` 匹配得分最高的结果。

In [ ]:
def cosine_similarity_manual(vec1, vec2):
    """Calculate the cosine similarity between two vectors."""
    vec1 = np.array(vec1, dtype=float)
    vec2 = np.array(vec2, dtype=float)


    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)


def find_similar_items(input_embedding, embeddings, threshold=0.5, top_k=2):
    """Find the most similar items based on cosine similarity."""
    
    # Calculate cosine similarity between the input embedding and all other embeddings
    similarities = [(index, cosine_similarity_manual(input_embedding, vec)) for index, vec in enumerate(embeddings)]
    
    # Filter out any similarities below the threshold
    filtered_similarities = [(index, sim) for index, sim in similarities if sim >= threshold]
    
    # Sort the filtered similarities by similarity score
    sorted_indices = sorted(filtered_similarities, key=lambda x: x[1], reverse=True)[:top_k]

    # Return the top-k most similar items
    return sorted_indices

In [ ]:
def find_matching_items_with_rag(df_items, item_descs):
   """Take the input item descriptions and find the most similar items based on cosine similarity for each description."""
   
   # Select the embeddings from the DataFrame.
   embeddings = df_items['embeddings'].tolist()

   
   similar_items = []
   for desc in item_descs:
      
      # Generate the embedding for the input item
      input_embedding = get_embeddings([desc])
    
      # Find the most similar items based on cosine similarity
      similar_indices = find_similar_items(input_embedding, embeddings, threshold=0.6)
      similar_items += [df_items.iloc[i] for i in similar_indices]
    
   return similar_items

### 分析模块

在本模块中，我们利用 `gpt-4o-mini` 分析输入图像，并提取详细描述、风格和类型等重要特征。分析通过一次简单的 API 调用完成，我们在调用中提供待分析图像的 URL，并要求模型识别相关特征。

为确保模型返回准确的结果，我们在提示中使用了一些特定技巧：

1. **指定输出格式**：我们指示模型返回一个具有预定义结构的 JSON 块，包含：
   - `items` (str[])：一个字符串列表，每个字符串代表一件服装的简洁标题，包括风格、颜色和性别。这些标题与原始数据库中的 `productDisplayName` 属性非常相似。
   - `category` (str)：最能代表给定单品的类别。模型从原始 styles dataframe 中所有唯一的 `articleTypes` 列表中进行选择。
   - `gender` (str)：表示该单品适用性别的标签。模型从 `[Men, Women, Boys, Girls, Unisex]` 选项中进行选择。

2. **清晰简洁的指令**：
   - 我们清楚地说明了单品标题应包含哪些内容以及输出格式应该是什么。输出应为 JSON 格式，但不带模型响应中通常包含的 `json` 标签。

3. **单样本（One Shot）示例**：
   - 为了进一步明确预期输出，我们向模型提供了一个示例输入描述和对应的示例输出。虽然这可能会增加使用的 token 数量（从而增加调用成本），但它有助于引导模型，并带来更好的整体表现。

通过遵循这种结构化的方法，我们旨在从 `gpt-4o-mini` 模型获取精确且有用的信息，以便进一步分析并集成到我们的数据库中。

In [ ]:
def analyze_image(image_base64, subcategories):
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": f"""Given an image of an item of clothing, analyze the item and generate a JSON output with the following fields: "items", "category", and "gender".
                           Use your understanding of fashion trends, styles, and gender preferences to provide accurate and relevant suggestions for how to complete the outfit.
                           The items field should be a list of items that would go well with the item in the picture. Each item should represent a title of an item of clothing that contains the style, color, and gender of the item.
                           The category needs to be chosen between the types in this list: {subcategories}.
                           You have to choose between the genders in this list: [Men, Women, Boys, Girls, Unisex]
                           Do not include the description of the item in the picture. Do not include the ```json ``` tag in the output.

                           Example Input: An image representing a black leather jacket.

                           Example Output: {{"items": ["Fitted White Women's T-shirt", "White Canvas Sneakers", "Women's Black Skinny Jeans"], "category": "Jackets", "gender": "Women"}}
                           """,
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}",
                },
                }
            ],
            }
        ]
    )
    # Extract relevant features from the response
    features = response.choices[0].message.content
    return features

### 使用示例图像测试提示

为了评估提示的有效性，我们从数据集中选取一些图像来加载并测试它。我们将使用 `"data/sample_clothes/sample_images"` 文件夹中的图像，确保涵盖多种风格、性别和类型。以下是选定的样本：

- `2133.jpg`：男士衬衫
- `7143.jpg`：女士衬衫
- `4226.jpg`：休闲男士印花 T 恤

通过使用这些多样化的图像测试提示，我们可以评估它准确分析各类服装和配饰并从中提取相关特征的能力。

我们需要一个工具函数，将 .jpg 图像编码为 base64

In [ ]:
import base64

def encode_image_to_base64(image_path):
    with open(image_path, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read())
        return encoded_image.decode('utf-8')

In [ ]:
# Set the path to the images and select a test image
image_path = "data/sample_clothes/sample_images/"
test_images = ["2133.jpg", "7143.jpg", "4226.jpg"]

# Encode the test image to base64
reference_image = image_path + test_images[0]
encoded_image = encode_image_to_base64(reference_image)

In [ ]:
# Select the unique subcategories from the DataFrame
unique_subcategories = styles_df['articleType'].unique()

# Analyze the image and return the results
analysis = analyze_image(encoded_image, unique_subcategories)
image_analysis = json.loads(analysis)

# Display the image and the analysis results
display(Image(filename=reference_image))
print(image_analysis)

接下来，我们处理图像分析的输出，并用它来过滤和展示数据集中的匹配单品。代码拆解如下：

1. **提取图像分析结果**：我们从 `image_analysis` 字典中提取单品描述、类别和性别。

2. **过滤数据集**：我们过滤 `styles_df` DataFrame，只保留与图像分析中的性别相匹配（或为中性）的单品，并排除与所分析图像属于同一类别的单品。

3. **查找匹配单品**：我们使用 `find_matching_items_with_rag` 函数，在过滤后的数据集中查找与所分析图像中提取的描述相匹配的单品。

4. **展示匹配单品**：我们创建一个 HTML 字符串来展示匹配单品的图像。我们使用单品 ID 构造图像路径，并将每张图像追加到 HTML 字符串中。最后，我们使用 `display(HTML(html))` 在 notebook 中渲染这些图像。

此单元格有效地演示了如何利用图像分析的结果来过滤数据集，并直观地展示与所分析图像特征相匹配的单品。

In [ ]:
# Extract the relevant features from the analysis
item_descs = image_analysis['items']
item_category = image_analysis['category']
item_gender = image_analysis['gender']


# Filter data such that we only look through the items of the same gender (or unisex) and different category
filtered_items = styles_df.loc[styles_df['gender'].isin([item_gender, 'Unisex'])]
filtered_items = filtered_items[filtered_items['articleType'] != item_category]
print(str(len(filtered_items)) + " Remaining Items")

# Find the most similar items based on the input item descriptions
matching_items = find_matching_items_with_rag(filtered_items, item_descs)

# Display the matching items (this will display 2 items for each description in the image analysis)
html = ""
paths = []
for i, item in enumerate(matching_items):
    item_id = item['id']
        
    # Path to the image file
    image_path = f'data/sample_clothes/sample_images/{item_id}.jpg'
    paths.append(image_path)
    html += f'<img src="{image_path}" style="display:inline;margin:1px"/>'

# Print the matching item description as a reminder of what we are looking for
print(item_descs)
# Display the image
display(HTML(html))

### 护栏

在使用 GPT-4o mini 等大语言模型（LLM）的场景中，“护栏”（guardrails）是指为确保模型输出保持在期望参数或边界之内而设置的机制或检查。这些护栏对于保持模型回复的质量和相关性至关重要，尤其是在处理复杂或微妙的任务时。

护栏的作用体现在以下几个方面：

1. **准确性**：有助于确保模型的输出准确，并与所提供的输入相关。
2. **一致性**：保持模型回复的一致性，尤其是在处理相似或相关的输入时。
3. **安全性**：防止模型生成有害、冒犯性或不当的内容。
4. **上下文相关性**：确保模型的输出与其所应用的特定任务或领域在上下文上相关。

在我们的案例中，我们使用 GPT-4o mini 分析时尚图像，并推荐能够与原始穿搭相搭配的单品。为了实现护栏，我们可以**优化结果**：在从 GPT-4o mini 获得初步建议后，我们可以将原始图像和建议的单品再发送回模型。然后，我们可以让 GPT-4o mini 评估每个建议的单品是否确实适合原始穿搭。

这让模型能够根据反馈或额外信息进行自我纠正并调整自己的输出。通过实施这些护栏并启用自我纠正，我们可以提升模型输出在时尚分析和推荐场景中的可靠性和实用性。

为此，我们编写了一个提示，要求 LLM 对建议的单品是否与原始穿搭相匹配这一问题给出简单的 "yes" 或 "no" 回答。这种二元回复有助于简化优化过程，并确保模型给出清晰且可操作的反馈。

In [ ]:
def check_match(reference_image_base64, suggested_image_base64):
    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": """ You will be given two images of two different items of clothing.
                            Your goal is to decide if the items in the images would work in an outfit together.
                            The first image is the reference item (the item that the user is trying to match with another item).
                            You need to decide if the second item would work well with the reference item.
                            Your response must be a JSON output with the following fields: "answer", "reason".
                            The "answer" field must be either "yes" or "no", depending on whether you think the items would work well together.
                            The "reason" field must be a short explanation of your reasoning for your decision. Do not include the descriptions of the 2 images.
                            Do not include the ```json ``` tag in the output.
                           """,
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{reference_image_base64}",
                },
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{suggested_image_base64}",
                },
                }
            ],
            }
        ],
        max_tokens=300,
    )
    # Extract relevant features from the response
    features = response.choices[0].message.content
    return features

最后，我们来确定上面识别出的单品中，哪些真正与这套穿搭相搭配。

In [ ]:
# Select the unique paths for the generated images
paths = list(set(paths))

for path in paths:
    # Encode the test image to base64
    suggested_image = encode_image_to_base64(path)
    
    # Check if the items match
    match = json.loads(check_match(encoded_image, suggested_image))
    
    # Display the image and the analysis results
    if match["answer"] == 'yes':
        display(Image(filename=path))
        print("The items match!")
        print(match["reason"])

我们可以看到，最初的候选单品列表经过了进一步优化，得到了一个与穿搭更加契合的精选结果。此外，模型还解释了为什么认为每件单品是合适的搭配，为了解其决策过程提供了有价值的见解。

### 结论

在本 Jupyter Notebook 中，我们探索了 GPT-4o mini 及其他机器学习技术在时尚领域的应用。我们演示了如何分析服装图像、提取相关特征，并利用这些信息查找能够与原始穿搭相搭配的单品。通过实施护栏和自我纠正机制，我们优化了模型的建议，确保其准确且与上下文相关。

这种方法在现实世界中有多种实际用途，包括：

1. **个性化购物助手**：零售商可以使用这项技术为顾客提供个性化的穿搭推荐，提升购物体验并提高顾客满意度。
2. **虚拟衣橱应用**：用户可以上传自己服装的图像来创建虚拟衣橱，并获得与现有衣物相搭配的新单品建议。
3. **时装设计与造型**：时装设计师和造型师可以使用此工具尝试不同的搭配和风格，简化创作过程。

不过，需要考虑的因素之一是**成本**。使用 LLM 和图像分析模型可能会产生费用，尤其是在大量使用时。考虑实施这些技术的成本效益非常重要。`gpt-4o-mini` 的价格为每 1000 个 token `$0.01`。对于一张 256px x 256px 的图像，费用合计为 `$0.00255`。

总体而言，本 notebook 为时尚与 AI 交叉领域的进一步探索和开发奠定了基础，为打造更加个性化、更加智能的时尚推荐系统打开了大门。